# Customer Segmentation using K-Means Clustering

This notebook demonstrates the complete machine learning workflow for segmenting mall customers using the Mall Customers dataset. The goal is to group customers into meaningful segments based on demographic and spending behavior.

## 1. Import Libraries and Set Project Path

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

current_path = Path.cwd()
project_root = current_path if (current_path / "src").exists() else current_path.parent
sys.path.append(str(project_root))

from src.config import DATA_FILE, FIGURE_DIR, MODEL_FILE, TRAINED_DATA_FILE
from src.data_loader import clean_dataset, load_dataset
from src.eda import generate_eda_plots
from src.model_training import (
    choose_best_cluster_count,
    create_cluster_descriptions,
    evaluate_cluster_range,
    train_final_model,
)
from src.preprocessing import prepare_features
from src.visualization import save_cluster_plot, save_elbow_plot, save_silhouette_plot

sns.set_theme(style="whitegrid")
print("Project root:", project_root)

## 2. Load the Dataset

The loader reads `Mall_Customers.csv`. If the file is not present, it tries public download links. If download also fails, it creates an equivalent sample dataset with 200 records.

In [ ]:
raw_data = load_dataset(DATA_FILE)
raw_data.head()

In [ ]:
raw_data.shape, raw_data.dtypes

## 3. Data Cleaning

The cleaning step removes duplicate rows, handles missing numeric values using medians, standardizes gender labels, and checks usable data types.

In [ ]:
cleaned_data = clean_dataset(raw_data)
print("Missing values after cleaning:")
print(cleaned_data.isnull().sum())
print("\nDuplicate rows:", cleaned_data.duplicated().sum())
cleaned_data.head()

## 4. Exploratory Data Analysis

In [ ]:
cleaned_data.describe()

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=cleaned_data, x="Gender", hue="Gender", palette="Set1", legend=False)
plt.title("Gender Distribution")
plt.show()

In [ ]:
cleaned_data[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].hist(
    figsize=(12, 8), bins=20
)
plt.suptitle("Feature Distributions")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    cleaned_data[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].corr(),
    annot=True,
    cmap="coolwarm",
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.pairplot(
    cleaned_data[["Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)"]],
    hue="Gender",
    diag_kind="hist",
)
plt.show()

## 5. Preprocessing

Gender is label encoded and the model features are scaled with StandardScaler.

In [ ]:
encoded_data, scaled_features, scaler = prepare_features(cleaned_data)
encoded_data.head()

## 6. Find the Optimal Number of Clusters

K-Means is trained for K values from 2 to 10. The elbow method helps select a practical number of clusters, while silhouette score provides a quality comparison.

In [ ]:
metrics_data = evaluate_cluster_range(scaled_features)
metrics_data

In [ ]:
best_cluster_count = choose_best_cluster_count(metrics_data)
print("Selected K using elbow method:", best_cluster_count)
print("Best silhouette-only K:", int(metrics_data.loc[metrics_data["Silhouette Score"].idxmax(), "Clusters"]))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(metrics_data["Clusters"], metrics_data["Inertia"], marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(metrics_data["Clusters"], metrics_data["Silhouette Score"], marker="o", color="crimson")
plt.title("Silhouette Scores")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

## 7. Train the Final K-Means Model

In [ ]:
final_model = train_final_model(scaled_features, best_cluster_count)
cluster_labels = final_model.predict(scaled_features)

clustered_data = encoded_data.copy()
clustered_data["Cluster"] = cluster_labels
cluster_profile = clustered_data.groupby("Cluster")[[
    "Gender_Encoded",
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)",
]].mean()
cluster_descriptions = create_cluster_descriptions(cluster_profile)
clustered_data["Segment_Name"] = clustered_data["Cluster"].map(
    lambda value: cluster_descriptions[int(value)].split(":")[0]
)
clustered_data.head()

## 8. Model Evaluation

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score

print("Silhouette Score:", silhouette_score(scaled_features, cluster_labels))
print("Inertia:", final_model.inertia_)
print("Davies-Bouldin Index:", davies_bouldin_score(scaled_features, cluster_labels))

## 9. Cluster Visualization

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=clustered_data,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Segment_Name",
    palette="tab10",
    s=75,
    edgecolor="black",
)
plt.title("Customer Segments")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.grid(True)
plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

## 10. Save Project Outputs

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TRAINED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)

generate_eda_plots(cleaned_data, FIGURE_DIR)
save_elbow_plot(metrics_data, FIGURE_DIR)
save_silhouette_plot(metrics_data, FIGURE_DIR)
save_cluster_plot(clustered_data, final_model, scaler, FIGURE_DIR)
clustered_data.to_csv(TRAINED_DATA_FILE, index=False)

final_metrics = {
    "Best Cluster Count": best_cluster_count,
    "Best Silhouette K": int(metrics_data.loc[metrics_data["Silhouette Score"].idxmax(), "Clusters"]),
    "Silhouette Score": silhouette_score(scaled_features, cluster_labels),
    "Inertia": final_model.inertia_,
    "Davies-Bouldin Index": davies_bouldin_score(scaled_features, cluster_labels),
}
model_package = {
    "model": final_model,
    "scaler": scaler,
    "feature_columns": [
        "Gender_Encoded",
        "Age",
        "Annual Income (k$)",
        "Spending Score (1-100)",
    ],
    "cluster_descriptions": cluster_descriptions,
    "cluster_profile": cluster_profile,
    "metrics": final_metrics,
}
joblib.dump(model_package, MODEL_FILE)
print("Outputs saved successfully.")

## Conclusion

The project successfully segments customers into business-friendly groups using K-Means clustering. The Streamlit app can use the saved model to predict a new customer's segment and display the EDA and clustering results.